# 03 基础拓扑构建示例

本 Notebook 将 `streets_cleaned_sample.parquet` 作为只读输入，生成道路要素邻接图、节点—边拓扑、节点度和构建报告。当前结构为二维基础拓扑，不包含道路等级、单双向、桥隧层级、车道和转向限制。

In [ ]:
# 运行环境检查：仅检查，不自动安装
import importlib.util

REQUIRED_MODULES = ['numpy', 'pandas', 'geopandas', 'shapely', 'pyarrow', 'matplotlib', 'momepy', 'libpysal', 'networkx']

missing_modules = [
    module
    for module in REQUIRED_MODULES
    if importlib.util.find_spec(module) is None
]

if missing_modules:
    raise ImportError(
        "当前环境缺少以下依赖："
        + ", ".join(missing_modules)
        + "\n可在终端或新的 Notebook Cell 中安装："
        + "\npython -m pip install numpy pandas geopandas shapely pyarrow matplotlib momepy libpysal networkx pyogrio"
    )

print("依赖检查通过。")


## 1. 路径、输入检查与绘图设置

In [ ]:
# ============================================================
# 清洗结果 → 基础节点—边拓扑与邻接图
#
# 输入：
#   道路清洗结果 streets_cleaned_sample.parquet
#
# 输出：
#   1. street_graph_0.parquet
#   2. nodes_graph_0.parquet
#   3. network_nodes_0.parquet
#   4. network_edges_0.parquet
#   5. network_build_report_0.csv
#   6. network_preview_0.png
#
# 说明：
#   - 不重新执行道路清洗
#   - 不覆盖 streets_0.parquet
#   - street_graph 表示道路要素之间的接触邻接关系
#   - nodes_graph 表示节点之间的空间权重关系
# ============================================================

from pathlib import Path
import datetime
import gc
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import shapely

import momepy as mm
import networkx as nx
from libpysal.graph import Graph


# ============================================================
# 0. 参数与路径
# ============================================================

TARGET_CRS = "EPSG:4547"
REGION_ID = 0

def locate_project_dir():
    """定位包含样例数据目录的项目根目录。"""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "03_小范围样例数据").exists():
            return candidate
    if cwd.name == "02_精简版代码":
        return cwd.parent
    return cwd

PROJECT_DIR = locate_project_dir()
OUTPUT_DIR = PROJECT_DIR / "04_示例输出" / "topology"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STREETS_PATH = PROJECT_DIR / "04_示例输出" / "streets_cleaned_sample.parquet"
STREET_GRAPH_PATH = OUTPUT_DIR / "street_graph_sample.parquet"
NODES_GRAPH_PATH = OUTPUT_DIR / "nodes_graph_sample.parquet"
NODES_GDF_PATH = OUTPUT_DIR / "network_nodes_sample.parquet"
EDGES_GDF_PATH = OUTPUT_DIR / "network_edges_sample.parquet"
REPORT_PATH = OUTPUT_DIR / "network_build_report.csv"


# ============================================================
# 1. 绘图和警告设置
# ============================================================

warnings.filterwarnings(
    "ignore",
    message=(
        r"Signature .*numpy\.longdouble.*"
        r"does not match any known type.*"
    ),
    category=UserWarning,
)

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = [
    "WenQuanYi Micro Hei",
    "WenQuanYi Zen Hei",
    "Microsoft YaHei",
    "SimHei",
    "DejaVu Sans",
]
mpl.rcParams["axes.unicode_minus"] = False


# ============================================================
# 2. 输入检查
# ============================================================

print("=" * 72)
print("V13道路结果 → 基础拓扑网络")
print("开始时间：", datetime.datetime.now())

if not STREETS_PATH.exists():

    raise FileNotFoundError(
        f"未找到V13道路文件：\n{STREETS_PATH}"
    )


streets = gpd.read_parquet(
    STREETS_PATH
)


if streets.crs is None:

    raise ValueError(
        "streets_0.parquet缺少CRS，"
        "请先确认其坐标系。"
    )


if streets.crs.to_string() != TARGET_CRS:

    print(
        f"坐标系转换：{streets.crs} → {TARGET_CRS}"
    )

    streets = streets.to_crs(
        TARGET_CRS
    )


# ============================================================


## 2. 整理标准化道路输入

In [ ]:
# 3. 基础几何检查
# ============================================================

input_feature_count = len(
    streets
)

missing_count = int(
    streets.geometry.isna().sum()
)

empty_count = int(
    streets.geometry.is_empty.sum()
)

valid_mask = (
    ~streets.geometry.isna()
    & ~streets.geometry.is_empty
    & streets.geometry.is_valid
)


streets = streets.loc[
    valid_mask
].copy()


# 强制二维
streets["geometry"] = shapely.force_2d(
    streets.geometry.array
)


# 仅保留线几何
streets = streets[
    streets.geometry.geom_type.isin([
        "LineString",
        "MultiLineString",
    ])
].copy()


# 拆分MultiLineString，但不重新清洗
streets = streets.explode(
    index_parts=False,
    ignore_index=True,
)


streets = streets[
    streets.geometry.geom_type
    == "LineString"
].copy()


streets = streets[
    streets.geometry.length > 0
].copy()


# 方向无关的完全重复检查
normalized_geometry = shapely.normalize(
    streets.geometry.array
)

normalized_wkb = shapely.to_wkb(
    normalized_geometry,
    hex=True,
)

duplicate_count = int(
    pd.Series(
        normalized_wkb
    ).duplicated(
        keep="first"
    ).sum()
)


# 仅在内存中去重，不覆盖V13正式输出
streets["_normalized_wkb"] = (
    normalized_wkb
)

streets = (
    streets
    .drop_duplicates(
        subset="_normalized_wkb"
    )
    .drop(
        columns="_normalized_wkb"
    )
    .reset_index(
        drop=True
    )
)


streets["street_id"] = np.arange(
    len(
        streets
    ),
    dtype=int,
)


print(
    "原始读取要素数：",
    f"{input_feature_count:,}"
)

print(
    "有效LineString数：",
    f"{len(streets):,}"
)

print(
    "检测到完全重复线：",
    f"{duplicate_count:,}"
)

print(
    "道路总长度：",
    f"{streets.geometry.length.sum() / 1000:.2f} km"
)


if streets.empty:

    raise RuntimeError(
        "整理后没有可用于构图的道路。"
    )


# ============================================================


## 3. 构建道路要素邻接图

In [ ]:
# 4. 构建 street_graph
#
# 每一条道路要素作为一个图节点；
# 发生空间接触的道路要素建立邻接关系。
# ============================================================

print("\n1/3 正在构建道路要素邻接图……")

street_graph = Graph.build_contiguity(
    streets,
    rook=False,
).assign_self_weight()


street_graph.to_parquet(
    STREET_GRAPH_PATH
)


print(
    "street_graph已输出：",
    STREET_GRAPH_PATH
)


# ============================================================


## 4. 构建节点—边拓扑并保存空间数据

In [ ]:
# 5. 构建节点—边 NetworkX 网络
# ============================================================

print("\n2/3 正在构建节点—边拓扑……")

nx_graph = mm.gdf_to_nx(
    streets,
    approach="primal",
    multigraph=True,
    directed=False,
    preserve_index=True,
)


nx_graph = mm.node_degree(
    nx_graph
)


nodes_gdf, edges_gdf, node_weights = (
    mm.nx_to_gdf(
        nx_graph,
        points=True,
        lines=True,
        spatial_weights=True,
    )
)


# 统一CRS
if nodes_gdf.crs is None:

    nodes_gdf = nodes_gdf.set_crs(
        TARGET_CRS,
        allow_override=True,
    )

elif nodes_gdf.crs.to_string() != TARGET_CRS:

    nodes_gdf = nodes_gdf.to_crs(
        TARGET_CRS
    )


if edges_gdf.crs is None:

    edges_gdf = edges_gdf.set_crs(
        TARGET_CRS,
        allow_override=True,
    )

elif edges_gdf.crs.to_string() != TARGET_CRS:

    edges_gdf = edges_gdf.to_crs(
        TARGET_CRS
    )


# 整理字段
nodes_gdf = nodes_gdf.reset_index()

edges_gdf = edges_gdf.reset_index()


if "degree" not in nodes_gdf.columns:

    nodes_gdf["degree"] = [
        int(
            nx_graph.degree[
                node
            ]
        )
        for node in nx_graph.nodes
    ]


nodes_gdf["node_id"] = np.arange(
    len(
        nodes_gdf
    ),
    dtype=int,
)


edges_gdf["edge_id"] = np.arange(
    len(
        edges_gdf
    ),
    dtype=int,
)


edges_gdf["length_m"] = (
    edges_gdf.geometry.length
)


# ============================================================
# 6. 保存节点、边和节点图
# ============================================================

nodes_gdf.to_parquet(
    NODES_GDF_PATH
)

edges_gdf.to_parquet(
    EDGES_GDF_PATH
)


nodes_graph = Graph.from_W(
    node_weights
)

nodes_graph.to_parquet(
    NODES_GRAPH_PATH
)


print(
    "节点空间要素：",
    NODES_GDF_PATH
)

print(
    "拓扑边空间要素：",
    EDGES_GDF_PATH
)

print(
    "nodes_graph：",
    NODES_GRAPH_PATH
)


# ============================================================


## 5. 统计连通分量与节点度

In [ ]:
# 7. 统计基础质量指标
# ============================================================

degree_series = pd.to_numeric(
    nodes_gdf["degree"],
    errors="coerce",
).fillna(
    0
)


degree1_count = int(
    (
        degree_series == 1
    ).sum()
)

degree3_count = int(
    (
        degree_series == 3
    ).sum()
)

degree4plus_count = int(
    (
        degree_series >= 4
    ).sum()
)


connected_components = list(nx.connected_components(nx_graph))


component_count = len(
    connected_components
)

largest_component_node_count = max(
    (
        len(
            component
        )
        for component in connected_components
    ),
    default=0,
)


largest_component_node_share = (
    largest_component_node_count
    / len(
        nodes_gdf
    )
    if len(
        nodes_gdf
    ) > 0
    else np.nan
)


report = pd.DataFrame([
    {
        "region_id": REGION_ID,

        "input_file": str(
            STREETS_PATH
        ),

        "input_feature_count": (
            input_feature_count
        ),

        "missing_geometry_count": (
            missing_count
        ),

        "empty_geometry_count": (
            empty_count
        ),

        "duplicate_geometry_count": (
            duplicate_count
        ),

        "network_input_line_count": int(
            len(
                streets
            )
        ),

        "network_input_length_km": float(
            streets.geometry.length.sum()
            / 1000
        ),

        "topology_node_count": int(
            len(
                nodes_gdf
            )
        ),

        "topology_edge_count": int(
            len(
                edges_gdf
            )
        ),

        "degree1_node_count": (
            degree1_count
        ),

        "degree3_node_count": (
            degree3_count
        ),

        "degree4plus_node_count": (
            degree4plus_count
        ),

        "connected_component_count": (
            component_count
        ),

        "largest_component_node_share": (
            largest_component_node_share
        ),
    }
])


report.to_csv(
    REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================


In [ ]:
# 9. 完成信息
# ============================================================

print("\n" + "=" * 72)
print("基础拓扑网络构建完成")

print(
    "道路邻接图：",
    STREET_GRAPH_PATH
)

print(
    "节点权重图：",
    NODES_GRAPH_PATH
)

print(
    "节点图层：",
    NODES_GDF_PATH
)

print(
    "拓扑边图层：",
    EDGES_GDF_PATH
)

print(
    "构建报告：",
    REPORT_PATH
)

print("\n构建结果：")
display(report)

gc.collect()


## 6. 图1：标准化道路中心线

In [ ]:
# ============================================================
# 图1｜标准化道路中心线
# ============================================================

STANDARD_ROADS_PNG = OUTPUT_DIR / "01_standardized_streets.png"

fig, ax = plt.subplots(figsize=(10, 9), dpi=180)
streets.plot(ax=ax, linewidth=0.55)
ax.set_title(
    f"标准化道路中心线\n{len(streets):,}个有效线部件，"
    f"{streets.geometry.length.sum() / 1000:.2f} km",
    fontsize=15,
    fontweight="bold",
    pad=14,
)
ax.text(
    0.01,
    0.01,
    "V13清洗结果；作为基础拓扑构建与下游空间计算的只读输入。",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.85},
)
ax.set_axis_off()
ax.set_aspect("equal")
plt.tight_layout()
fig.savefig(STANDARD_ROADS_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("标准化道路图已输出：", STANDARD_ROADS_PNG)


## 7. 图2：节点—边网络与节点度

In [ ]:
# ============================================================
# 图2｜节点—边网络与节点度
#
# 内容：
#   - 绘制清洗后道路对应的拓扑边
#   - 按节点度设置节点大小
#   - 标注连接度最高的主要交叉节点
#
# 输入：
#   network_nodes_0.parquet
#   network_edges_0.parquet
#
# 输出：
#   02_node_network_and_degree.png
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib as mpl


# ============================================================
# 0. 路径与参数
# ============================================================

TARGET_CRS = "EPSG:4547"
NODES_PATH = NODES_GDF_PATH
EDGES_PATH = EDGES_GDF_PATH
OUTPUT_PNG = OUTPUT_DIR / "02_node_network_and_degree.png"


# 标注连接度最高的节点数量
TOP_NODE_LABEL_COUNT = 8


mpl.rcParams["font.family"] = "sans-serif"

mpl.rcParams["font.sans-serif"] = [
    "WenQuanYi Micro Hei",
    "WenQuanYi Zen Hei",
    "Microsoft YaHei",
    "SimHei",
    "DejaVu Sans",
]

mpl.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. 读取节点和拓扑边
# ============================================================

for path in [
    NODES_PATH,
    EDGES_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"未找到网络结果文件：\n{path}\n"
            "请先运行基础节点—边拓扑构建Cell。"
        )


nodes = gpd.read_parquet(
    NODES_PATH
)

edges = gpd.read_parquet(
    EDGES_PATH
)


if nodes.crs is None:

    nodes = nodes.set_crs(
        TARGET_CRS,
        allow_override=True,
    )

elif nodes.crs.to_string() != TARGET_CRS:

    nodes = nodes.to_crs(
        TARGET_CRS
    )


if edges.crs is None:

    edges = edges.set_crs(
        TARGET_CRS,
        allow_override=True,
    )

elif edges.crs.to_string() != TARGET_CRS:

    edges = edges.to_crs(
        TARGET_CRS
    )


if "degree" not in nodes.columns:

    raise KeyError(
        "节点图层中没有degree字段。"
    )


nodes["degree"] = pd.to_numeric(
    nodes["degree"],
    errors="coerce",
).fillna(
    0
).astype(
    int
)


if "node_id" not in nodes.columns:

    nodes["node_id"] = np.arange(
        len(nodes),
        dtype=int,
    )


print(
    "拓扑节点数：",
    f"{len(nodes):,}"
)


print(
    "拓扑边数：",
    f"{len(edges):,}"
)


print(
    "最高节点度：",
    int(
        nodes["degree"].max()
    )
)


# ============================================================
# 2. 节点大小
# ============================================================

# 低度节点保持较小，高度节点明显放大
node_sizes = (
    2.5
    + np.power(
        nodes["degree"].to_numpy(),
        1.7,
    )
)


node_sizes = np.clip(
    node_sizes,
    3,
    90,
)


# ============================================================
# 3. 识别主要交叉节点
# ============================================================

top_nodes = (
    nodes
    .sort_values(
        [
            "degree",
            "node_id",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .head(
        TOP_NODE_LABEL_COUNT
    )
    .copy()
)


# ============================================================
# 4. 绘图
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        10,
    ),
    dpi=180,
)


# 拓扑边
edges.plot(
    ax=ax,
    linewidth=0.40,
    alpha=0.55,
)


# 全部节点：大小和颜色均表示degree
nodes.plot(
    ax=ax,
    column="degree",
    markersize=node_sizes,
    alpha=0.80,
    legend=True,
    legend_kwds={
        "label": "节点度",
        "shrink": 0.65,
    },
)


# 主要交叉节点再次放大
top_nodes.plot(
    ax=ax,
    markersize=(
        25
        + top_nodes[
            "degree"
        ].to_numpy() ** 2
    ),
    facecolor="none",
    linewidth=1.0,
)


# 标注高连接度节点
for _, row in top_nodes.iterrows():

    point = row.geometry

    ax.annotate(
        (
            f"节点 {int(row['node_id'])}\n"
            f"degree={int(row['degree'])}"
        ),
        xy=(
            point.x,
            point.y,
        ),
        xytext=(
            5,
            5,
        ),
        textcoords="offset points",
        fontsize=7.5,
        bbox={
            "boxstyle": "round,pad=0.2",
            "facecolor": "white",
            "alpha": 0.75,
            "linewidth": 0.4,
        },
    )


ax.set_title(
    "基础节点—边网络与节点度",
    fontsize=16,
    fontweight="bold",
    pad=16,
)


ax.text(
    0.01,
    0.01,
    (
        f"节点：{len(nodes):,}个　"
        f"拓扑边：{len(edges):,}条\n"
        "节点大小与颜色表示连接道路数量，"
        "标注为连接度最高的主要交叉节点。"
    ),
    transform=ax.transAxes,
    fontsize=9.5,
    va="bottom",
    bbox={
        "boxstyle": "round,pad=0.35",
        "facecolor": "white",
        "alpha": 0.85,
        "linewidth": 0.5,
    },
)


ax.set_axis_off()
ax.set_aspect(
    "equal"
)


plt.tight_layout()


fig.savefig(
    OUTPUT_PNG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)


plt.show()


print(
    "图片已输出：",
    OUTPUT_PNG
)